# VARC Prefix Ablation Study

Compare 4 prefix variants (Baseline / MLP / Transformer / Mamba) under identical TTT conditions on ARC-AGI evaluation tasks.

**Before running:** Upload the entire VARC repo to your Google Drive under `My Drive/VARC/`.

## 0. Setup: Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Change this if your VARC folder is at a different path in Drive
VARC_ROOT = '/content/drive/MyDrive/Colab Notebooks/VARC'
assert os.path.isdir(VARC_ROOT), f"VARC not found at {VARC_ROOT}, please check your Drive path"

# Copy to local disk for faster I/O
!cp -r "{VARC_ROOT}" /content/VARC

os.chdir('/content/VARC')
print(f"Working directory: {os.getcwd()}")
!ls

Working directory: /content/VARC
outputs


In [ ]:
import torch
print(f"Colab PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

# Use Colab's pre-installed PyTorch — do NOT reinstall torch/torchvision
!pip install -q timm==1.0.12 einops wandb numpy huggingface_hub datasets diffusers safetensors
!pip install -q packaging setuptools wheel ninja
!pip install -q causal-conv1d --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation

Colab PyTorch: 2.10.0+cu128, CUDA: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.3 MB/s eta 0:00:00


In [ ]:
# Verify installations
import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

from mamba_ssm import Mamba
print("mamba-ssm OK")

from src.ARC_HybridViT import HybridARCViT
print("HybridARCViT import OK")

## 1. Download Pretrained ViT Checkpoint & Build Augmented Data

In [ ]:
!mkdir -p saves/offline_train_ViT

# Download pretrained ViT checkpoint from HuggingFace
from huggingface_hub import snapshot_download
snapshot_download('VisionARC/offline_train_ViT', local_dir='saves/offline_train_ViT')
print("Checkpoint downloaded:")
!ls -lh saves/offline_train_ViT/

In [ ]:
# Build augmented TTT data (takes ~1 min)
!python augment_data.py

## 2. Ablation Study: Prefix Variant Comparison

Compare 4 prefix configurations under identical TTT conditions:

| Variant | Prefix Type | Description |
|---------|------------|-------------|
| Baseline | None | Original VARC ViT |
| MLP prefix | 2-layer wide MLP | Parameter-matched to Mamba |
| Transformer prefix | 2-layer self-attention | Standard attention blocks |
| Mamba prefix | 2-layer Mamba/SSM | State-space model |

All variants load the **same pretrained ViT checkpoint** (`checkpoint_best.pt`). Prefix blocks are randomly initialized. TTT updates all parameters.

Settings aligned with VARC paper: **100 TTT epochs**, lr=3e-4, batch=8, cosine scheduler, 10 attempts, ttt-num-each=2.

In [ ]:
import os, glob, time

VARC_ROOT = '/content/drive/MyDrive/Colab Notebooks/VARC'

ALL_TASKS_400 = [
    'af24b4cc', 'e1d2900e', '903d1b4a', '4e469f39', 'b1fc8b8e', '2c737e39', '992798f6', '00576224', '48131b3c', '60a26a3e',
    '59341089', '31d5ba1a', 'e633a9e5', '62ab2642', '73c3b0d8', 'c663677b', 'c48954c1', '08573cc6', '136b0064', '929ab4e9',
    '5b526a93', 'ef26cbf6', 'fafd9572', '67c52801', 'ad7e01d0', '506d28a5', '27a77e38', 'd492a647', '72a961c9', 'fd4b2b02',
    'bf89d739', 'f5aa3634', 'b942fd60', 'd282b262', '9772c176', 'ed74f2f2', '184a9768', '94133066', '256b0a75', 'e681b708',
    'ce8d95cc', '817e6c09', '7d18a6fb', '1da012fc', '310f3251', 'bf699163', '917bccba', '551d5bf1', 'b457fec5', '50a16a69',
    '7953d61e', '9a4bb226', 'c97c0139', 'd4b1c2b1', '1d398264', '29700607', '8597cfd7', 'a59b95c0', '2f0c5170', 'bd14c3bf',
    '9caba7c3', 'e6de6e8f', 'da515329', '31adaf00', 'f5c89df1', 'be03b35f', '833dafe3', '6ea4a07e', '2546ccf6', '21f83797',
    '696d4842', 'd94c3b52', '4ff4c9da', '3979b1a8', 'bf32578f', 'd304284e', 'c62e2108', 'b0722778', 'd19f7514', '358ba94e',
    'd017b73f', '4c177718', 'b7999b51', 'e345f17b', 'e4075551', '50aad11f', '66e6c45b', 'c074846d', '0b17323b', '4b6b68e5',
    '84db8fc4', 'ff72ca3e', '8ee62060', '52fd389e', 'ae58858e', 'fea12743', '0f63c0b9', 'e99362f0', '195ba7dc', 'f3cdc58f',
    'a8610ef7', 'e760a62e', 'aa300dc3', 'ea9794b1', 'e41c6fd3', '5d2a5c43', 'e66aafb8', 'ca8de6ea', '19bb5feb', '7c8af763',
    'e872b94a', '6f473927', 'ac605cbb', 'ac3e2b04', '0e671a1a', 'ac0c5833', 'fb791726', '351d6448', 'ce039d91', '45bbe264',
    '332efdb3', 'c64f1187', '5b6cbef5', '1d0a4b61', '42918530', '7bb29440', '3a301edc', '896d5239', '505fff84', 'cfb2ce5a',
    '140c817e', '69889d6e', '20818e16', '9b2a60aa', '626c0bcc', 'a57f2f04', '477d2879', '05a7bcf2', '81c0276b', 'ba9d41b8',
    'e133d23d', '604001fa', '3ee1011a', '85b81ff1', '17b80ad2', '9b365c51', 'e7dd8335', '2a5f8217', '712bf12e', '84f2aca1',
    'ac2e8ecf', 'e2092e0c', '33b52de3', '5833af48', '319f2597', 'aa18de87', 'cb227835', 'e74e1818', '15663ba9', 'b4a43f3b',
    '281123b4', 'fc754716', 'e5790162', '94414823', '642d658d', '96a8c0cd', '2697da3f', 'e9b4f6fc', 'bcb3040b', '55783887',
    '1acc24af', '981571dc', '705a3229', '1c02dbbe', 'ca8f78db', '1e97544e', '92e50de0', 'e57337a4', '4852f2fa', '7d1f7ee8',
    'e1baa8a4', '14754a24', '62b74c02', '7d419a02', '94be5b80', '68b67ca3', '2072aba6', 'fe9372f3', '137f0df0', 'c6e1b8da',
    '16b78196', '1c0d0a4b', 'f0afb749', 'de493100', '1990f7a8', '423a55dc', '2753e76c', 'f21745ec', 'bc4146bd', '79fb03f4',
    '3d31c5b3', 'c35c1b4c', 'cf133acc', 'da2b0fe3', '15696249', '0c9aba6e', 'e7a25a18', 'd5c634a2', '414297c0', '009d5c81',
    '0becf7df', 'f3e62deb', '58743b76', '9b4c17c4', '891232d6', '4e45f183', 'c7d4e6ad', 'a04b2602', 'd37a1ef5', '25094a63',
    '0c786b71', 'f3b10344', 'b7f8a4d8', 'b7cb93ac', 'b15fca0b', '1a6449f1', '67636eac', 'f823c43c', '27f8ce4f', 'fd096ab6',
    '0a1d4ef5', '6a11f6da', '8fbca751', '6ad5bdfd', '3ed85e70', '09c534e7', '642248e4', '9f27f097', '50f325b5', '88207623',
    '45737921', '11e1fe23', 'aee291af', '90347967', 'e7b06bea', '03560426', 'e7639916', 'e21a174a', '4aab4007', 'c658a4bd',
    '5783df64', '1c56ad9f', 'c1990cce', '93c31fbe', '5af49b42', '7e02026e', '2685904e', 'c8b7cc0f', '15113be4', 'b20f7c8b',
    '575b1a71', 'e0fb7511', '3b4c2228', '32e9702f', 'ccd554ac', 'af22c60d', 'bbb1b8b6', 'f9d67f8b', '0d87d2a6', 'ed98d772',
    '9ddd00f0', '070dd51e', '9356391f', '4acc7107', '47996f11', '8dae5dfc', 'e5c44e8f', 'e9bb6954', 'dc2aa30b', 'd2acf2cb',
    '292dd178', 'f9a67cb5', '20981f0e', '12997ef3', '103eff5b', '770cc55f', '0692e18c', '8719f442', 'e88171ec', '95a58926',
    '639f5a19', '40f6cd08', '3f23242b', 'd47aa2ff', '5b692c0f', 'a934301b', 'a3f84088', '72207abc', '73182012', '0a2355a6',
    'a680ac02', '58e15b12', '64a7c07e', 'e9c9d9a1', '12eac192', 'dc2e9a9d', 'ecaa0ec1', '4cd1b7b2', '7c9b52a0', '3391f8c0',
    '9c56f360', '0607ce86', '97239e3d', 'a406ac07', 'baf41dbf', 'c3202e5a', '13713586', '42a15761', 'e619ca6e', 'e69241bd',
    'd56f2372', '5207a7b5', '8ba14f53', 'b0f4d537', 'aa4ec2a5', '3490cc26', '9def23fe', 'd4c90558', '12422b43', '99306f82',
    '516b51b7', 'cad67732', 'f4081712', '212895b5', '67b4a34d', '845d6e51', '66f2d22f', '73ccf9c2', 'f83cb3f6', 'd931c21c',
    '17cae0c1', '22a4bbc2', '0934a4d8', '8cb8642d', '1e81d6f9', '85fa5666', '0bb8deee', '55059096', '4364c1c4', '4f537728',
    'aab50785', '9110e3c5', '762cd429', '7039b2d7', 'bb52a14b', 'ea959feb', 'dd2401ed', '48f8583b', '2b01abd0', 'b7fb29bc',
    '8b28cd80', '782b5218', '5a5a2103', '759f3fd3', 'e9ac8c9e', '9c1e755f', '37d3e8b2', '9bebae7a', '1a2e2828', '34b99a2b',
    '60c09cac', 'f45f5ca7', '7ee1c6ea', '54db823b', '00dbd492', 'b9630600', '2037f2c7', 'c87289bb', '692cd3b6', '79369cc6',
    'c92b942c', '6df30ad6', 'a096bf4d', 'e78887d1', '5ffb2104', '5289ad53', '8a371977', 'f0df5ff0', 'df8cc377', 'cd3c21df',
    '963f59bc', '3194b014', '93b4f4b3', '2c0b0aff', '456873bc', '8e2edd66', 'f8be4b64', 'e95e3d8e', '695367ec', '18419cfa',
]

# --------------- Ablation config ---------------
ABLATION_TASKS = ALL_TASKS_400[:100]  # first 100 tasks; set to ALL_TASKS_400 for full eval
TTT_EPOCHS = 100
NUM_ATTEMPTS = 2

VARIANT_NAMES = [
    ("Baseline ViT",       "ablation_baseline"),
    ("MLP prefix",         "ablation_mlp_prefix"),
    ("Transformer prefix", "ablation_transformer_prefix"),
    ("Mamba prefix",       "ablation_mamba_prefix"),
]

# Ensure output dirs exist on both local and Drive
os.makedirs("outputs", exist_ok=True)
os.makedirs(f"{VARC_ROOT}/outputs", exist_ok=True)

print(f"Ablation tasks: {len(ABLATION_TASKS)}")
print(f"TTT epochs: {TTT_EPOCHS}")
print(f"Checkpoint: saves/offline_train_ViT/checkpoint_best.pt")
print(f"Drive sync path: {VARC_ROOT}/outputs/")

Ablation tasks: 100
TTT epochs: 100
Checkpoint: saves/offline_train_ViT/checkpoint_best.pt
Drive sync path: /content/drive/MyDrive/Colab Notebooks/VARC/outputs/


In [ ]:
import shutil

# Uncomment to clear ALL previous ablation results:
# for _, save_name in VARIANT_NAMES:
#     for i in range(NUM_ATTEMPTS):
#         shutil.rmtree(f"outputs/{save_name}_attempt_{i}", ignore_errors=True)
#     print(f"Cleared {save_name}")

print("Uncomment the lines above and re-run this cell to clear old results.")

Uncomment the lines above and re-run this cell to clear old results.


### 2a. TTT — Baseline ViT (no prefix)

Original VARC architecture, no prefix blocks added.

In [ ]:
import os, glob, time

SAVE_NAME = "ablation_baseline"

# Restore previous results from Drive (survives runtime restarts)
for i in range(NUM_ATTEMPTS):
    src = f"{VARC_ROOT}/outputs/{SAVE_NAME}_attempt_{i}"
    dst = f"outputs/{SAVE_NAME}_attempt_{i}"
    if os.path.isdir(src) and not os.path.isdir(dst):
        os.makedirs(dst, exist_ok=True)
        !cp -rn "{src}"/* "{dst}"/ 2>/dev/null

def is_task_done(task_id):
    return all(
        os.path.exists(f"outputs/{SAVE_NAME}_attempt_{i}/{task_id}_predictions.json")
        for i in range(NUM_ATTEMPTS)
    )

tasks = ABLATION_TASKS
done = sum(1 for t in tasks if is_task_done(t))
total = len(tasks)
print(f"[Baseline] Progress: {done}/{total} completed, {total - done} remaining.")

start_time = time.time()
tasks_run = 0

for idx, task in enumerate(tasks):
    if is_task_done(task):
        continue

    tasks_run += 1
    elapsed = time.time() - start_time
    if tasks_run > 1:
        eta_min = elapsed / (tasks_run - 1) * (total - done - tasks_run + 1) / 60
        print(f"\n[Baseline {done+tasks_run}/{total}] {task}  ETA: {eta_min:.0f} min")
    else:
        print(f"\n[Baseline {done+tasks_run}/{total}] {task}")

    !python test_time_train_ARC.py \
        --epochs {TTT_EPOCHS} \
        --depth 10 \
        --batch-size 32 \
        --image-size 64 \
        --patch-size 2 \
        --learning-rate 3e-4 \
        --weight-decay 0 \
        --embed-dim 512 \
        --num-heads 8 \
        --num-colors 12 \
        --lr-scheduler "cosine" \
        --resume-skip-task-token \
        --num-attempts 10 \
        --no-compile \
        --ttt-num-each {NUM_ATTEMPTS} \
        --resume-checkpoint "saves/offline_train_ViT/checkpoint_best.pt" \
        --architecture "vit" \
        --train-split "eval_color_permute_ttt_9/{task}" \
        --data-root "raw_data/ARC-AGI" \
        --eval-split "eval_color_permute_ttt_9/{task}" \
        --eval-save-name "{SAVE_NAME}"

    !cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

done_final = sum(1 for t in tasks if is_task_done(t))
print(f"\n[Baseline] Done: {done_final}/{total}")
!cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

[Baseline] Progress: 93/100 completed, 7 remaining.

[Baseline 94/100] 52fd389e
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Start loading data...
Finish loading!!
Start loading data...
Finish loading!!
Total training examples: 153
Resuming from checkpoint: saves/offline_train_ViT/checkpoint_best.pt
Patch size: 2, sequence length: 1024
Randomly initialized parameters (not in checkpoint): ['task_token_embed.weight']
Starting test-time training attempt 1/2...
Using automatic mixed precision (AMP) training

epoch=0 | train_loss=4.3489 | train_acc=0.0000 | epoch_time=3.8s | eta_total=00h06m16s | lr=0.000000

epoch=1 | train_loss=2.5364 | train_acc=0.0000 | epoch_time=2.1s | eta_total=00h04m48s | lr=0.000030

epoch=2 | train_loss=0.6959 | tra

### 2b. TTT — ViT + MLP Prefix

2-layer wide-MLP prefix, parameter-matched to Mamba (~4.4M extra params).

In [ ]:
import os, glob, time

SAVE_NAME = "ablation_mlp_prefix"

for i in range(NUM_ATTEMPTS):
    src = f"{VARC_ROOT}/outputs/{SAVE_NAME}_attempt_{i}"
    dst = f"outputs/{SAVE_NAME}_attempt_{i}"
    if os.path.isdir(src) and not os.path.isdir(dst):
        os.makedirs(dst, exist_ok=True)
        !cp -rn "{src}"/* "{dst}"/ 2>/dev/null

def is_task_done(task_id):
    return all(
        os.path.exists(f"outputs/{SAVE_NAME}_attempt_{i}/{task_id}_predictions.json")
        for i in range(NUM_ATTEMPTS)
    )

tasks = ABLATION_TASKS
done = sum(1 for t in tasks if is_task_done(t))
total = len(tasks)
print(f"[MLP prefix] Progress: {done}/{total} completed, {total - done} remaining.")

start_time = time.time()
tasks_run = 0

for idx, task in enumerate(tasks):
    if is_task_done(task):
        continue

    tasks_run += 1
    elapsed = time.time() - start_time
    if tasks_run > 1:
        eta_min = elapsed / (tasks_run - 1) * (total - done - tasks_run + 1) / 60
        print(f"\n[MLP prefix {done+tasks_run}/{total}] {task}  ETA: {eta_min:.0f} min")
    else:
        print(f"\n[MLP prefix {done+tasks_run}/{total}] {task}")

    !python test_time_train_ARC.py \
        --epochs {TTT_EPOCHS} \
        --depth 10 \
        --batch-size 32 \
        --image-size 64 \
        --patch-size 2 \
        --learning-rate 3e-4 \
        --weight-decay 0 \
        --embed-dim 512 \
        --num-heads 8 \
        --num-colors 12 \
        --lr-scheduler "cosine" \
        --resume-skip-task-token \
        --num-attempts 10 \
        --no-compile \
        --ttt-num-each {NUM_ATTEMPTS} \
        --resume-checkpoint "saves/offline_train_ViT/checkpoint_best.pt" \
        --architecture "vit_hybrid_mamba" \
        --prefix-type "mlp" --mamba-depth 2 --mamba-d-state 16 --mamba-d-conv 4 --mamba-expand 2 \
        --train-split "eval_color_permute_ttt_9/{task}" \
        --data-root "raw_data/ARC-AGI" \
        --eval-split "eval_color_permute_ttt_9/{task}" \
        --eval-save-name "{SAVE_NAME}"

    !cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

done_final = sum(1 for t in tasks if is_task_done(t))
print(f"\n[MLP prefix] Done: {done_final}/{total}")
!cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

Streaming output truncated to the last 5000 lines.

epoch=85 | train_loss=0.0001 | train_acc=0.9804 | epoch_time=2.8s | eta_total=00h00m39s | lr=0.000020

epoch=86 | train_loss=0.0001 | train_acc=0.9608 | epoch_time=2.8s | eta_total=00h00m37s | lr=0.000018

epoch=87 | train_loss=0.0002 | train_acc=0.9608 | epoch_time=2.8s | eta_total=00h00m34s | lr=0.000015

epoch=88 | train_loss=0.0001 | train_acc=0.9706 | epoch_time=2.8s | eta_total=00h00m31s | lr=0.000013

epoch=89 | train_loss=0.0001 | train_acc=0.9902 | epoch_time=2.8s | eta_total=00h00m28s | lr=0.000011

epoch=90 | train_loss=0.0001 | train_acc=0.9706 | epoch_time=2.8s | eta_total=00h00m25s | lr=0.000009

epoch=91 | train_loss=0.0001 | train_acc=1.0000 | epoch_time=2.8s | eta_total=00h00m22s | lr=0.000007

epoch=92 | train_loss=0.0001 | train_acc=0.9804 | epoch_time=2.8s | eta_total=00h00m19s | lr=0.000006

epoch=93 | train_loss=0.0001 | train_acc=0.9608 | epoch_time=2.8s | eta_total=00h00m17s | lr=0.000004

epoch=94 | train_loss

### 2c. TTT — ViT + Transformer Prefix

2-layer self-attention prefix blocks (~3.2M extra params).

In [ ]:
import os, glob, time

SAVE_NAME = "ablation_transformer_prefix"

for i in range(NUM_ATTEMPTS):
    src = f"{VARC_ROOT}/outputs/{SAVE_NAME}_attempt_{i}"
    dst = f"outputs/{SAVE_NAME}_attempt_{i}"
    if os.path.isdir(src) and not os.path.isdir(dst):
        os.makedirs(dst, exist_ok=True)
        !cp -rn "{src}"/* "{dst}"/ 2>/dev/null

def is_task_done(task_id):
    return all(
        os.path.exists(f"outputs/{SAVE_NAME}_attempt_{i}/{task_id}_predictions.json")
        for i in range(NUM_ATTEMPTS)
    )

tasks = ABLATION_TASKS
done = sum(1 for t in tasks if is_task_done(t))
total = len(tasks)
print(f"[Transformer prefix] Progress: {done}/{total} completed, {total - done} remaining.")

start_time = time.time()
tasks_run = 0

for idx, task in enumerate(tasks):
    if is_task_done(task):
        continue

    tasks_run += 1
    elapsed = time.time() - start_time
    if tasks_run > 1:
        eta_min = elapsed / (tasks_run - 1) * (total - done - tasks_run + 1) / 60
        print(f"\n[Transformer prefix {done+tasks_run}/{total}] {task}  ETA: {eta_min:.0f} min")
    else:
        print(f"\n[Transformer prefix {done+tasks_run}/{total}] {task}")

    !python test_time_train_ARC.py \
        --epochs {TTT_EPOCHS} \
        --depth 10 \
        --batch-size 32 \
        --image-size 64 \
        --patch-size 2 \
        --learning-rate 3e-4 \
        --weight-decay 0 \
        --embed-dim 512 \
        --num-heads 8 \
        --num-colors 12 \
        --lr-scheduler "cosine" \
        --resume-skip-task-token \
        --num-attempts 10 \
        --no-compile \
        --ttt-num-each {NUM_ATTEMPTS} \
        --resume-checkpoint "saves/offline_train_ViT/checkpoint_best.pt" \
        --architecture "vit_hybrid_mamba" \
        --prefix-type "transformer" --mamba-depth 2 \
        --train-split "eval_color_permute_ttt_9/{task}" \
        --data-root "raw_data/ARC-AGI" \
        --eval-split "eval_color_permute_ttt_9/{task}" \
        --eval-save-name "{SAVE_NAME}"

    !cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

done_final = sum(1 for t in tasks if is_task_done(t))
print(f"\n[Transformer prefix] Done: {done_final}/{total}")
!cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

### 2d. TTT — ViT + Mamba Prefix

2-layer Mamba/SSM prefix blocks (~4.4M extra params).

In [ ]:
import os, glob, time

SAVE_NAME = "ablation_mamba_prefix"

for i in range(NUM_ATTEMPTS):
    src = f"{VARC_ROOT}/outputs/{SAVE_NAME}_attempt_{i}"
    dst = f"outputs/{SAVE_NAME}_attempt_{i}"
    if os.path.isdir(src) and not os.path.isdir(dst):
        os.makedirs(dst, exist_ok=True)
        !cp -rn "{src}"/* "{dst}"/ 2>/dev/null

def is_task_done(task_id):
    return all(
        os.path.exists(f"outputs/{SAVE_NAME}_attempt_{i}/{task_id}_predictions.json")
        for i in range(NUM_ATTEMPTS)
    )

tasks = ABLATION_TASKS
done = sum(1 for t in tasks if is_task_done(t))
total = len(tasks)
print(f"[Mamba prefix] Progress: {done}/{total} completed, {total - done} remaining.")

start_time = time.time()
tasks_run = 0

for idx, task in enumerate(tasks):
    if is_task_done(task):
        continue

    tasks_run += 1
    elapsed = time.time() - start_time
    if tasks_run > 1:
        eta_min = elapsed / (tasks_run - 1) * (total - done - tasks_run + 1) / 60
        print(f"\n[Mamba prefix {done+tasks_run}/{total}] {task}  ETA: {eta_min:.0f} min")
    else:
        print(f"\n[Mamba prefix {done+tasks_run}/{total}] {task}")

    !python test_time_train_ARC.py \
        --epochs {TTT_EPOCHS} \
        --depth 10 \
        --batch-size 8 \
        --image-size 64 \
        --patch-size 2 \
        --learning-rate 3e-4 \
        --weight-decay 0 \
        --embed-dim 512 \
        --num-heads 8 \
        --num-colors 12 \
        --lr-scheduler "cosine" \
        --resume-skip-task-token \
        --num-attempts 10 \
        --no-compile \
        --ttt-num-each {NUM_ATTEMPTS} \
        --resume-checkpoint "saves/offline_train_ViT/checkpoint_best.pt" \
        --architecture "vit_hybrid_mamba" \
        --prefix-type "mamba" --mamba-depth 2 --mamba-d-state 16 --mamba-d-conv 4 --mamba-expand 2 \
        --train-split "eval_color_permute_ttt_9/{task}" \
        --data-root "raw_data/ARC-AGI" \
        --eval-split "eval_color_permute_ttt_9/{task}" \
        --eval-save-name "{SAVE_NAME}"

    !cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

done_final = sum(1 for t in tasks if is_task_done(t))
print(f"\n[Mamba prefix] Done: {done_final}/{total}")
!cp -r outputs/{SAVE_NAME}_attempt_* "{VARC_ROOT}/outputs/" 2>/dev/null

## 3. Analyze & Compare Results

Run at any time to check progress. Generates per-variant HTML reports and a summary CSV for plotting.

In [ ]:
import os
import glob

empty_json_files = []
# Search for all prediction json files in the outputs directory
for filepath in glob.glob("outputs/**/*_predictions.json", recursive=True):
    # Check if the file is empty (0 bytes)
    if os.path.getsize(filepath) == 0:
        empty_json_files.append(filepath)

if empty_json_files:
    print(f"Found {len(empty_json_files)} empty/corrupted JSON files. Deleting them...")
    for f in empty_json_files:
        os.remove(f)
        print(f"Deleted: {f}")
    print("\nCleanup complete! You can now re-run the analysis cell.")
else:
    print("No empty JSON files found. The issue might be a malformed JSON with content.")


Found 12 empty/corrupted JSON files. Deleting them...
Deleted: outputs/ablation_mamba_prefix_attempt_0/bf89d739_predictions.json
Deleted: outputs/outputs/ablation_mamba_prefix_attempt_0/bf89d739_predictions.json
Deleted: outputs/outputs/outputs/ablation_mamba_prefix_attempt_0/bf89d739_predictions.json
Deleted: outputs/outputs/outputs/outputs/ablation_mamba_prefix_attempt_0/bf89d739_predictions.json
Deleted: outputs/outputs/outputs/outputs/outputs/ablation_mamba_prefix_attempt_0/bf89d739_predictions.json
Deleted: outputs/outputs/outputs/outputs/outputs/outputs/ablation_baseline_attempt_1/917bccba_predictions.json
Deleted: outputs/outputs/outputs/outputs/outputs/ablation_baseline_attempt_1/917bccba_predictions.json
Deleted: outputs/outputs/outputs/outputs/ablation_baseline_attempt_1/917bccba_predictions.json
Deleted: outputs/outputs/outputs/ablation_baseline_attempt_1/917bccba_predictions.json
Deleted: outputs/outputs/ablation_baseline_attempt_1/917bccba_predictions.json
Deleted: outputs

In [ ]:
import os, glob, subprocess, re, csv

# Restore outputs from Drive if local copies are missing (e.g. after runtime restart)
drive_outputs = f"{VARC_ROOT}/outputs"
if os.path.isdir(drive_outputs):
    !cp -rn "{drive_outputs}"/* outputs/ 2>/dev/null
    print("Restored outputs from Drive.")

summary = []

for label, save_name in VARIANT_NAMES:
    print(f"\n{'=' * 60}")
    print(f"  {label}  ({save_name})")
    print(f"{'=' * 60}")

    attempt_dirs = [f"outputs/{save_name}_attempt_{i}" for i in range(NUM_ATTEMPTS)]
    counts = []
    for d in attempt_dirs:
        if os.path.isdir(d):
            c = len(glob.glob(f"{d}/*_predictions.json"))
            counts.append(c)
            print(f"  {d}: {c} tasks")
        else:
            print(f"  {d}: NOT FOUND")
            counts.append(0)

    if min(counts) == 0:
        summary.append({"variant": label, "tasks": 0, "pass1": None, "pass2": None, "oracle": None})
        continue

    output_roots = ",".join(attempt_dirs)
    html_name = f"analysis_{save_name}.html"
    result = subprocess.run(
        ["python", "analysis.py",
         "--output-root", output_roots,
         "--task-type", "ARC-AGI",
         "--html-output", html_name],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    p1 = re.search(r"Final Pass@1 Score: ([\d.]+)", result.stdout)
    p2 = re.search(r"Final Pass@2 Score: ([\d.]+)", result.stdout)
    orc = re.search(r"Final Oracle Score: ([\d.]+)", result.stdout)
    summary.append({
        "variant": label,
        "tasks": min(counts),
        "pass1": float(p1.group(1)) if p1 else None,
        "pass2": float(p2.group(1)) if p2 else None,
        "oracle": float(orc.group(1)) if orc else None,
    })

# ---- Summary table ----
print(f"\n{'=' * 60}")
print("  ABLATION SUMMARY")
print(f"{'=' * 60}")
print("{:<25} {:>5}  {:>7}  {:>7}  {:>7}".format("Variant", "Tasks", "Pass@1", "Pass@2", "Oracle"))
print("-" * 60)
for row in summary:
    p1 = f"{row['pass1']:.4f}" if row["pass1"] is not None else "  N/A"
    p2 = f"{row['pass2']:.4f}" if row["pass2"] is not None else "  N/A"
    orc = f"{row['oracle']:.4f}" if row["oracle"] is not None else "  N/A"
    print("{:<25} {:>5}  {:>7}  {:>7}  {:>7}".format(row['variant'], row['tasks'], p1, p2, orc))

# ---- Save CSV ----
csv_path = "ablation_results.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["variant", "tasks", "pass1", "pass2", "oracle"])
    w.writeheader()
    w.writerows(summary)
print(f"\nSaved to {csv_path}")

# ---- Copy to Drive ----
!cp -r outputs/ "{VARC_ROOT}/outputs/" 2>/dev/null
!cp ablation_results.csv "{VARC_ROOT}/" 2>/dev/null
!cp analysis_ablation_*.html "{VARC_ROOT}/" 2>/dev/null
print("Results copied to Drive.")

Restored outputs from Drive.

  Baseline ViT  (ablation_baseline)
  outputs/ablation_baseline_attempt_0: 100 tasks
  outputs/ablation_baseline_attempt_1: 100 tasks
100
Oracle rank 1: 58.5. Cumulative: 0.5850
Oracle rank 2: 3.0. Cumulative: 0.6150
Oracle rank 3: 2.0. Cumulative: 0.6350
Oracle rank 8: 1.0. Cumulative: 0.6450
Oracle rank 16: 0.5. Cumulative: 0.6500
Oracle rank 25: 1.0. Cumulative: 0.6600
Oracle rank 79: 1.0. Cumulative: 0.6700
Oracle rank 125: 1.0. Cumulative: 0.6800
Oracle rank 362: 1.0. Cumulative: 0.6900
Final Oracle Score: 0.6900
Final Pass@1 Score: 0.5850
Final Pass@2 Score: 0.6150
HTML visualization saved to /content/VARC/analysis_ablation_baseline.html


  MLP prefix  (ablation_mlp_prefix)
  outputs/ablation_mlp_prefix_attempt_0: 100 tasks
  outputs/ablation_mlp_prefix_attempt_1: 100 tasks
100
Oracle rank 1: 58.5. Cumulative: 0.5850
Oracle rank 2: 4.0. Cumulative: 0.6250
Oracle rank 3: 1.0. Cumulative: 0.6350
Oracle rank 4: 1.0. Cumulative: 0.6450
Oracle rank 11: 0